# Vallidate the Results of the Multi Unit-Regression 

The multi-unit regression uses building, parcel, and zillow data to calculate the units for multi-family homes when the provided unit is missing. To validate this data first we validate the output corresponds with the expected values. Then these values are validated at the census tract level to ensure the hosting capacity analysis is being completed on reliable data. 

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt

In [2]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

# read in ca boundary to clip zillow
ca_boundary = gpd.read_file("/../../capstone/electrigrid/data/ca_state_boundary/CA_State.shp").to_crs(epsg=4326)

# import multifamily homes with units 
multi_zillow_w_units = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/final_zillow_w_volume/multi_zillow_w_units2.parquet").to_crs(epsg=4326)

# import multifamily homes with units 
keep_outliers_multi_zillow_w_units = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/final_zillow_w_volume/keep_outliers_multi_zillow_w_units.parquet").to_crs(epsg=4326)

# import single family homes with  adjusted units 
single_condo_zillow_w_units = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/final_zillow_w_volume/single_condo_zillow_w_units.parquet").to_crs(epsg=4326)

In [3]:
# clip Zillow to california 
zillow = zillow.clip(ca_boundary)

### Validate the outputs of the unit data with the original Zillow dataset

In [4]:
print(f"The orginal Zillow data includes ", zillow.shape[0], "points")

The orginal Zillow data includes  9071177 points


In [5]:
# investigate the unit totals
multi_zillow = zillow[(zillow['type'] == 'Multi') & (zillow['code'] != 'RR106')]
single_zillow = zillow[zillow['type'] == 'Single']
condo_multi = zillow[(zillow['type'] == 'Multi') & (zillow['code'] == 'RR106')]

In [6]:
# condo data is crazy!
zillow[(zillow['type'] == 'Multi') & (zillow['code'] == 'RR106')]

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry
7784446,Multi,1995.0,1.0,None,None,O,1.0,244626.0,living,636.0,8146806,06073013607,h366,SDGE,RR106,POINT (-116.93578 32.74326)
7784429,Multi,1995.0,2.0,None,None,None,1.0,265114.0,living,889.0,8146789,06073013607,h366,SDGE,RR106,POINT (-116.93578 32.74326)
7832395,Multi,1979.0,2.0,None,None,None,1.0,187736.0,living,960.0,8198473,06073003404,h569,SDGE,RR106,POINT (-117.10065 32.71337)
7832396,Multi,1979.0,2.0,None,None,None,1.0,187736.0,living,960.0,8198474,06073003404,h569,SDGE,RR106,POINT (-117.10065 32.71337)
7832397,Multi,1979.0,2.0,None,None,None,1.0,187736.0,living,960.0,8198475,06073003404,h569,SDGE,RR106,POINT (-117.10065 32.71337)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1164566,Multi,1973.0,2.0,fossil,None,None,NaN,192994.0,living,925.0,1339300,06023001001,415,PGE/SCE,RR106,POINT (-124.08995 40.86908)
1164565,Multi,1973.0,2.0,fossil,None,I,NaN,223686.0,living,927.0,1339299,06023001001,415,PGE/SCE,RR106,POINT (-124.08971 40.86910)
1185930,Multi,NaN,NaN,None,None,None,NaN,NaN,None,NaN,1368297,06023001200,8,PGE/SCE,RR106,POINT (-124.08289 40.89861)
1188126,Multi,NaN,NaN,None,None,None,NaN,NaN,None,NaN,1370734,06023010502,8,PGE/SCE,RR106,POINT (-124.10033 40.93822)


In [7]:
print(f"There are ", multi_zillow['unit'].sum() + single_zillow.shape[0] + condo_multi.shape[0], "units before the regression of multifamily homes.")

There are  11859239.0 units before the regression of multifamily homes.


In [8]:
print(f"The regressed Zillow data is", multi_zillow_w_units.shape[0] + single_condo_zillow_w_units.shape[0] ,"points")

The regressed Zillow data is 9940002 points


In [9]:
print(f"The number of homes calculated in California by unit (when dropping outliers):", single_condo_zillow_w_units['total_unit'].sum() + multi_zillow_w_units['total_unit'].sum())

The number of homes calculated in California by unit (when dropping outliers): 20473622.0


In [18]:
print(f'The number of homes calculated in California by unit (when keeping outliers): ', single_condo_zillow_w_units['total_unit'].sum() + keep_outliers_multi_zillow_w_units['total_unit'].sum())

The number of homes calculated in California by unit (when keeping outliers):  12505709.0


# Investigate differences

In [10]:
print(f"Maximum units for multi-family homes before the regression",multi_zillow['unit'].max())

Maximum units for multi-family homes before the regression 6857.0


In [11]:
print(f"Maximum units for the multi-family homes after the regression", multi_zillow_w_units['total_unit'].max())

Maximum units for the multi-family homes after the regression 32508.0


In [12]:
multi_zillow_w_units['total_unit'].sum()

11064428.0